# 09 — Gate AB-3: the manifest freeze (the pre-registration artifact)

Kernel `y2y-geo`. Zero solves. Mirrors the parent's `11_gate3_freeze`: derives the **14
formulations** — (S0–S5) × {ssp585, ssp245} + the two crossed formulations s1x/s3x at ssp585 —
from `scenarios_ab_v1.json` by the parent §3.1 recipes on the AB stack, at the **primary budget
level** decided by 08's nesting test, and writes `spec/manifest.csv` + `spec/manifest_freeze.sha256`.
Schema = parent §9 + the AB columns (`config`, `extent_id`, `mirror_spec_version`, `lock_rule`,
`budget_semantics`, `budget_level`, `floor_g`, `pipeline_git_sha`, `pipeline_module_sha256`).

**Code provenance (D-AB / v0.4 flag 4):** the freeze records the working tree's git SHA and asserts
the five pipeline modules (`config.py`, `leverage_core.py`, `ensemble_core.py`, `prioritizr_core.R`,
`mga_core.R`) are unmodified relative to HEAD — commit them first, or the freeze refuses. Notebooks
and specs may be dirty (they are the record being written).

Reference formulation `s0_ssp585_theta5`: its anchor/twin/MGA artifacts from AB-1/AB-2 at the
primary level are pointed to, not re-solved (parent convention).

In [ ]:
# ---- bootstrap ----------------------------------------------------------------------------------------
import hashlib, importlib, json, pathlib, subprocess, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc
for _m in (config, lc):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
SPEC, AUDIT_OBJ, RUNS = HERE / "spec", HERE / "audit" / "audit_objects_ab", HERE / "runs" / "ab_l"
AB = config.AB_HANDOFF_DIR
SC = json.loads((SPEC / "scenarios_ab_v1.json").read_text())
LV = json.loads((SPEC / "ab_budget_levels_v1.json").read_text())
V2 = json.loads((SPEC / "gate_ab2_verdicts.json").read_text())
CONSTS = json.loads((AUDIT_OBJ / "audit_constants.json").read_text())
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())
LEVEL = V2["nesting"]["primary_level"]
BUDGET = LV["levels"][LEVEL]
T_S4 = float(SC["S4_carbon"]["targets"]["irrecoverable_carbon_m_soc"])
TH_S4 = SC["_meta"]["s4_ladder"]["chosen_theta"]
REG_S4 = f"theta{str(TH_S4).rstrip('0').rstrip('.')}_places"
VERDICT_RULE = V2["verdict_rule_hash"]
assert VERDICT_RULE == "v2_8db80fed1c702638"
REALIZATION = {"ssp585_2071_2100": None,
               "ssp245_2071_2100": AB / "climate_realizations" / "macrorefugia_245_2071_2100.tif"}
sha245 = hashlib.sha256(REALIZATION["ssp245_2071_2100"].read_bytes()).hexdigest()
print(f"primary budget level {LEVEL}: {BUDGET['budget_cells']:,} cells ({100*BUDGET['budget_pct']:.1f}%), additions {BUDGET['additions_cells']:,} | "
      f"S4 regime {REG_S4} (t {T_S4}) | verdict rule {VERDICT_RULE} | audit {CONSTS['created_utc'][:19]}")

# ---- code provenance: git SHA + pipeline modules unmodified vs HEAD ----------------------------------------
MODULES = ["config.py", "leverage_core.py", "ensemble_core.py", "prioritizr_core.R", "mga_core.R"]
git = lambda *a: subprocess.run(["git", *a], cwd=ROOT, capture_output=True, text=True)
GIT_SHA = git("rev-parse", "HEAD").stdout.strip()
dirty = [m for m in MODULES if git("diff", "--quiet", "HEAD", "--", m).returncode != 0]
assert not dirty, f"pipeline module(s) modified vs HEAD -- commit before freezing: {dirty}"
MOD_SHA = {m: hashlib.sha256((ROOT / m).read_bytes()).hexdigest() for m in MODULES}
print(f"pipeline pinned at {GIT_SHA[:12]} (modules clean vs HEAD)")

In [ ]:
# ---- derive the 14 formulations (parent 11 recipes, AB stack, primary level) --------------------------------
def _norm(d, what):
    tot = sum(d.values()); assert abs(tot - 1.0) < 5e-3, f"{what} sums to {tot:.6f}"
    return {k: v / tot for k, v in d.items()}
S0 = SC["S0_balanced"]
recipes = {}
for name in ("S0_balanced", "S1_core_habitat", "S2_connectivity", "S3_biodiversity", "S4_carbon"):
    s = SC[name]
    recipes[name.split("_")[0].lower()] = dict(
        scenario_name=name, shares=_norm(s["block_shares"], name),
        within={b: _norm(m, f"{name}/{b}") for b, m in s["within_block"].items()},
        targets=s["targets"], extra={}, regime=REG_S4 if name == "S4_carbon" else "theta5_amount")
recipes["s5"] = dict(scenario_name="S5_intactness", shares=_norm(S0["block_shares"], "S5"),
                     within={b: _norm(m, f"S5/{b}") for b, m in S0["within_block"].items()}, targets=S0["targets"],
                     extra={"human_modification": 10.0}, regime="theta5_amount")
for sid, base in (("s1x", "S1_core_habitat"), ("s3x", "S3_biodiversity")):
    recipes[sid] = dict(scenario_name=f"{base.split('_')[0]}xCarbonRegime", shares=_norm(SC[base]["block_shares"], sid),
                        within={b: _norm(m, f"{sid}/{b}") for b, m in S0["within_block"].items()},
                        targets={"irrecoverable_carbon_m_soc": T_S4}, extra={}, regime=REG_S4)

def derive(recipe, climate):
    lp = {"climate_type_macrorefugia": REALIZATION[climate]} if REALIZATION[climate] is not None else None
    d = lc.scenario_weights(recipe["shares"], within_block=recipe["within"], targets=recipe["targets"],
                            handoff_dir=AB, layer_paths=lp)
    w = {r.feature: round(r.w, 6) for r in d.itertuples()}
    w.update(recipe["extra"])
    return w, {r.feature: round(r.intended_share, 6) for r in d.itertuples()}

forms = []
for climate in ("ssp585_2071_2100", "ssp245_2071_2100"):
    for sid in ("s0", "s1", "s2", "s3", "s4", "s5"):
        w, prof = derive(recipes[sid], climate)
        forms.append(dict(sid=sid, climate=climate, r=recipes[sid], w=w, prof=prof))
for sid in ("s1x", "s3x"):
    w, prof = derive(recipes[sid], "ssp585_2071_2100")
    forms.append(dict(sid=sid, climate="ssp585_2071_2100", r=recipes[sid], w=w, prof=prof))
assert len(forms) == 14
for c in forms:
    c["formulation_id"] = f"{c['sid']}_{c['climate'].split('_')[0]}_{c['r']['regime'].split('_')[0]}"
# cross-check: 05's frozen S0-S4 vectors (585 and 245) must be reproduced here
for c in forms:
    if c["sid"] in ("s0", "s1", "s2", "s3", "s4"):
        key = "weights_ssp245" if c["climate"].startswith("ssp245") else "weights"
        ref = SC[c["r"]["scenario_name"]][key]
        assert all(abs(c["w"][f] - ref[f]) < 1e-5 for f in ref), f"{c['formulation_id']}: weights differ from scenarios_ab_v1"
print(pd.DataFrame([{"formulation_id": c["formulation_id"], **{k.split("_")[-1][:10]: v for k, v in c["w"].items()}} for c in forms]).round(3).to_string(index=False))

In [ ]:
# ---- write spec/manifest.csv + freeze hash ----------------------------------------------------------------
REF = "s0_ssp585_theta5"
ref_dir = RUNS / LEVEL / REF
for art in ("anchor/run_summary.json", "twin/run_summary.json", "mga_g05.tif", "mga_guard_g05.tif", "formulation_meta.json"):
    assert (ref_dir / art).exists(), f"reference artifact missing at the primary level: {art}"
rows, now = [], datetime.now(timezone.utc).isoformat()
for c in forms:
    hashes = dict(CONSTS["layer_sha256"])
    if REALIZATION[c["climate"]] is not None:
        hashes["climate_type_macrorefugia"] = sha245
    is_ref = c["formulation_id"] == REF
    rel = lambda p: str(p.relative_to(ROOT))
    rows.append(dict(
        formulation_id=c["formulation_id"], scenario_id=c["sid"], scenario_name=c["r"]["scenario_name"],
        climate_level=c["climate"], carbon_regime=c["r"]["regime"],
        config="ab_l", extent_id=EXTENT["extent_id"], mirror_spec_version="v0.14.1",
        lock_rule="pa_mask (inherited, D-AB2)", budget_semantics="locked + X*unlocked (D-AB5 v2)",
        budget_level=LEVEL, budget_cells=BUDGET["budget_cells"], budget_pct=BUDGET["budget_pct"],
        weight_vector=json.dumps(c["w"]), target_vector=json.dumps(c["r"]["targets"]),
        influence_profile_intended=json.dumps(c["prof"]),
        k_requested=50, band_gap_g=0.05, applied_band_g=0.02, floor_g=0.05, opt_gap=1e-4, numeric_focus=2,
        dust_rule_version="1e-9 / AB re-run 2026-09-03 (0 cells zeroed)", estimator="mga_maxham_v1",
        verdict_rule=VERDICT_RULE, solver="gurobi",
        seed_policy="deterministic (MGA warm-start chain; no RNG)",
        input_layer_hashes=json.dumps(hashes),
        anchor_ref=rel(ref_dir / "anchor") if is_ref else "", twin_ref=rel(ref_dir / "twin") if is_ref else "",
        mga_ref=rel(ref_dir) if is_ref else "",
        pipeline_git_sha=GIT_SHA, pipeline_module_sha256=json.dumps(MOD_SHA),
        created_utc=now, frozen=True))
M = pd.DataFrame(rows)
assert M.formulation_id.is_unique and len(M) == 14
out = SPEC / "manifest.csv"
M.to_csv(out, index=False)
digest = hashlib.sha256(out.read_bytes()).hexdigest()
(SPEC / "manifest_freeze.sha256").write_text(f"{digest}  manifest.csv\n")
print(f"FROZEN: {out.relative_to(ROOT)} (14 formulations at level {LEVEL}) | sha256 {digest[:16]}...")
print("commit spec/manifest.csv + spec/manifest_freeze.sha256 -- that commit IS the pre-registration")

## → next

Commit the freeze, then `10_ab4_ensemble.ipynb` (R). Log the freeze in `spec/results_log.md` (R6).